# Lesson 1.4 — Iterating to Improve Output Quality

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Treat the first response as a draft, not a final answer
- Give targeted feedback to refine outputs
- Use examples and constraints to steer revisions

**You will produce:** A documented 3-round refinement log saved as a .md file

## Concepts

The first answer is rarely the best answer. **Iteration** is the normal workflow: review, give specific feedback, and ask for a revision.

Effective feedback is **specific**: 'Make it shorter and remove jargon' beats 'make it better'. You can also:
- Provide an example of the style you want.
- Add a constraint you forgot ('keep under 100 words').
- Point to exactly what to fix ('the second paragraph is too formal').

This maps to Domain 1: *Iterate prompts to improve output quality* (and sets up troubleshooting in Phase 5).

> **Reference:** [Anthropic Docs - Prompt engineering overview](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(".env", usecwd=True)

if not env_path:
    raise FileNotFoundError("Could not find .env from the notebook's working directory")

load_dotenv(env_path)
print("API key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

def ask_claude(prompt, system=None, model="claude-sonnet-5-5", max_tokens=4000):
    """Send a prompt to Claude. Falls back to a printed mock if no API key is set,
    so this notebook always runs. Set ANTHROPIC_API_KEY in your environment to use the real API.
    NOTE: Verify current model names at https://docs.anthropic.com (they change over time)."""
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("[No ANTHROPIC_API_KEY found - showing a MOCK response so the lesson still runs.]\n")
        return "[MOCK CLAUDE OUTPUT]\nThis is a placeholder. With a real key, Claude's answer appears here."
    try:
        import anthropic
        client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
        kwargs = {"model": model, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        # Some models return a thinking block first, so join only the text blocks
        text = "".join(b.text for b in resp.content if b.type == "text")
        if not text.strip():
            print(f"[Empty text reply (stop_reason={resp.stop_reason}). Thinking may have used the whole max_tokens budget - raise max_tokens.]")
        return text
    except Exception as e:
        print(f"[Could not reach the API: {e}. Showing a mock response.]\n")
        return "[MOCK CLAUDE OUTPUT] Placeholder response."

print("Helper ready. Call ask_claude('your prompt here').")

## Code-Along: Step 1 — Round 1: a first draft

Generate something to improve.

In [ ]:
draft1 = ask_claude('Write a thank-you email to a client after a project wrapped up.')
print(draft1)

## Code-Along: Step 2 — Round 2 and 3: targeted feedback

Each revision gives one or two specific instructions.

In [ ]:
draft2 = ask_claude(f'Revise this email to be under 80 words and warmer. Keep a clear subject line.\n\n{draft1}')
print('--- ROUND 2 ---'); print(draft2)
draft3 = ask_claude(f'Now add one specific sentence inviting a short feedback call next week.\n\n{draft2}')
print('\n--- ROUND 3 ---'); print(draft3)

## Exercise

Run your own 3-round refinement and save the log. Fill in the TODO feedback lines.

In [ ]:
r1 = ask_claude('Draft a LinkedIn post announcing a webinar on Agentic workflow.')
print(r1)
fb2 = 'I want a shorter version, with a more formal tone, DO NOT add shorter version suggestions or tips)'
print('######## REVISION 2 ########')
r2 = ask_claude(f'Revise based on this feedback: {fb2}\n\n{r1}')
print(r2)
fb3 = 'make it more shorter and precise, DO NOT add shorter version suggestions or tips, ADD emojis)'
print('######## REVISION 3 ########')
r3 = ask_claude(f'Revise based on this feedback: {fb3}\n\n{r2}')
print(r3)
r3 = ask_claude(f'Revise based on this feedback: {fb3}\n\n{r2}')
log = f'# Refinement log\n\n## Round 1\n{r1}\n\n## Feedback\n{fb2}\n\n## Round 2\n{r2}\n\n## Feedback\n{fb3}\n\n## Round 3\n{r3}'
with open('refinement_log.md','w',encoding='utf-8') as f:
    f.write(log)
print('Saved refinement_log.md')

## Key Takeaways

- The first output is a draft; plan to iterate
- Specific feedback produces better revisions than vague requests
- Examples and constraints steer the style and length
- Keeping a refinement log helps you reuse what worked

---

**Next:** [Lesson 1.5 — Adapting Prompts by Task Type](./lesson-1.5-adapting-by-task-type.ipynb)